# 확률 기반 은퇴 시뮬레이터 (Colab 노트북)

위에서부터 순서대로 실행하면 됩니다 (`런타임 > 모두 실행`).

- **1단계**: 모델 코드 설치 (수정할 필요 없음)
- **2단계**: 가구 정보 입력 ← 여기 숫자만 바꿔서 실험
- **3단계 이후**: 결과, 선택지 비교, 금리 시나리오, 쇼크 테스트, 민감도, 검증

모든 금액 단위는 **만원**, 가정값은 **임시값**입니다.

## English guide
This notebook runs the probabilistic retirement simulator end to end.

1. Select `Runtime > Run all`.
2. Edit only the **Step 2 (2단계) input cell**. All amounts are in **KRW 10,000 (만원)**; e.g. KRW 500M = `50_000`.
3. Key inputs: `ME` / `SPOUSE` (age, sex "M"/"F", National Pension monthly amount, private pension), `LIQUID_ASSETS`, `STOCK_WEIGHT`, `SPENDING` (annual), housing (`HOUSE_OFFICIAL` assessed value, `HOUSE_MARKET`), business income (`ME_BIZ`).
4. Outputs: depletion probability with 95% CI, pension-claiming comparison (Step 4), rate/inflation scenarios (5), shock tests (6), sensitivity (7), validation (8), tax strategies (9), gifting trade-off (10), pension timing × taxes (11).

Assumptions are placeholders; tax rules are simplified (Korea, as of 2026-09). Not tax or investment advice.
© 2026 Minyoung Kang. All rights reserved.

## 1단계. 모델 코드 설치
아래 셀들은 `retire_sim` 패키지 파일을 만듭니다. 실행만 하면 됩니다.

In [ ]:
import os
os.makedirs('retire_sim', exist_ok=True)

In [ ]:
%%writefile retire_sim/__init__.py
"""확률 기반 은퇴 시뮬레이터 (v1 파이프라인 뼈대)."""


In [ ]:
%%writefile retire_sim/config.py
"""입력 스키마. 모든 금액은 '현재가치 기준 만원/년'."""
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class Person:
    age: int
    sex: str                      # "M" | "F"
    nps_monthly: float = 0.0      # 국민연금 예상 월액(정상수령 기준, 만원)
    nps_normal_age: int = 65      # 정상 수령 개시 나이
    nps_start_age: int = 65       # 실제 수령 개시 나이(조기/연기 반영)
    private_pension_annual: float = 0.0   # 사적연금 연액(명목 고정, 만원)
    private_pension_start: int = 60
    private_pension_years: int = 20


@dataclass
class Household:
    members: List[Person]
    liquid_assets: float          # 금융자산 합계(만원)
    stock_weight: float = 0.5     # 주식 비중(연 1회 리밸런싱)
    annual_spending: float = 3600 # 부부 기준 연 생활비(만원, 현재가치)
    survivor_spending_ratio: float = 0.7  # 한 명 사망 시 생활비 비율


@dataclass
class EconomyAssumptions:
    stock_mu: float = 0.06        # 주식 기대 산술수익률
    stock_sigma: float = 0.18
    bond_mu: float = 0.03
    bond_sigma: float = 0.05
    rho_stock_bond: float = 0.0
    # 물가: 이산 OU  pi_{t+1} = pi_t + kappa(theta - pi_t) + sigma*eps
    infl_theta: float = 0.02
    infl_kappa: float = 0.3
    infl_sigma: float = 0.01
    infl_init: float = 0.025


@dataclass
class NPSRules:
    """국민연금 조기/연기 조정률 — 개발 전 최신 기준 재확인 필요."""
    early_cut_per_year: float = 0.06
    defer_add_per_year: float = 0.072
    max_early_years: int = 5
    max_defer_years: int = 5


@dataclass
class CareShock:
    """간병비 점프: 나이 >= start_age인 생존자에게 연 lam 확률로 발생."""
    enabled: bool = True
    start_age: int = 75
    lam: float = 0.03
    cost_median: float = 2000     # 발생 시 연 비용 중앙값(만원)
    cost_log_sigma: float = 0.5
    duration_years: int = 3


@dataclass
class SimConfig:
    n_paths: int = 10_000
    max_age: int = 110
    seed: Optional[int] = 42
    economy: EconomyAssumptions = field(default_factory=EconomyAssumptions)
    nps: NPSRules = field(default_factory=NPSRules)
    care: CareShock = field(default_factory=CareShock)


In [ ]:
%%writefile retire_sim/mortality.py
"""사망률 모듈.

v1은 Gompertz 근사(임시값). 실제 사용 시 통계청(KOSIS) 완전생명표 CSV를
load_life_table()로 불러와 교체한다. CSV 형식: age,qx_M,qx_F
"""
import numpy as np
import csv

# 임시 Gompertz 파라미터 mu(x) = a*exp(b*x) — 한국 대략 수준에 맞춘 값, 실데이터로 교체 필요
_GOMPERTZ = {"M": (1.65e-5, 0.100), "F": (4.28e-6, 0.108)}  # 60세 기대여명 남 23.5 / 여 29.5년에 보정


def gompertz_qx(max_age: int = 110) -> dict:
    ages = np.arange(max_age + 1)
    out = {}
    for sex, (a, b) in _GOMPERTZ.items():
        H = a / b * (np.exp(b * (ages + 1)) - np.exp(b * ages))  # 1년 누적위험
        q = 1 - np.exp(-H)
        q[-1] = 1.0
        out[sex] = np.clip(q, 0, 1)
    return out


def load_life_table(path: str, max_age: int = 110) -> dict:
    qM, qF = np.ones(max_age + 1), np.ones(max_age + 1)
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            a = int(row["age"])
            if a <= max_age:
                qM[a], qF[a] = float(row["qx_M"]), float(row["qx_F"])
    return {"M": qM, "F": qF}


def life_expectancy(qx: np.ndarray, age: int) -> float:
    surv = np.cumprod(1 - qx[age:])
    return float(surv.sum() + 0.5)


def simulate_alive(qx: np.ndarray, start_age: int, T: int, n: int, rng) -> np.ndarray:
    """alive[p, t] : t년 시작 시점 생존 여부 (t=0은 현재)."""
    ages = np.minimum(start_age + np.arange(T), len(qx) - 1)
    die = rng.random((n, T)) < qx[ages]
    alive = np.ones((n, T + 1), dtype=bool)
    alive[:, 1:] = np.cumprod(~die, axis=1).astype(bool)
    return alive


In [ ]:
%%writefile retire_sim/economy.py
"""경제 시나리오 생성기: 상관 GBM(주식·채권) + OU 물가."""
import numpy as np
from .config import EconomyAssumptions


def generate(e: EconomyAssumptions, T: int, n: int, rng) -> dict:
    cov = np.array([[e.stock_sigma**2, e.rho_stock_bond * e.stock_sigma * e.bond_sigma],
                    [e.rho_stock_bond * e.stock_sigma * e.bond_sigma, e.bond_sigma**2]])
    L = np.linalg.cholesky(cov)
    Z = rng.standard_normal((n, T, 2)) @ L.T
    stock = np.exp(e.stock_mu - 0.5 * e.stock_sigma**2 + Z[..., 0]) - 1
    bond = np.exp(e.bond_mu - 0.5 * e.bond_sigma**2 + Z[..., 1]) - 1

    infl = np.empty((n, T))
    pi = np.full(n, e.infl_init)
    for t in range(T):
        pi = pi + e.infl_kappa * (e.infl_theta - pi) + e.infl_sigma * rng.standard_normal(n)
        infl[:, t] = pi
    cpi = np.cumprod(1 + infl, axis=1)          # t년 말 물가지수
    cpi = np.concatenate([np.ones((n, 1)), cpi], axis=1)  # cpi[:, t] = t년 시작 시점
    return {"stock": stock, "bond": bond, "infl": infl, "cpi": cpi}


In [ ]:
%%writefile retire_sim/economy_v2.py
"""경제 시나리오 v2: Vasicek 단기금리 + OU 물가 + 초과수익 주식, 정확 이산화.

- 단기금리 r (Vasicek):   dr = k_r(th_r - r)dt + s_r dW_r
- 물가상승률 pi (OU):      dpi = k_p(th_p - pi)dt + s_p dW_p
- 주식 로그수익 = r_t + ERP - s_s^2/2 + s_s * Z_s
- 채권: 만기 D년 무이표채를 1년 보유 후 (D-1)년물로 매도하는 롤링 펀드.
  가격은 Vasicek 해석해 P(tau, r) = exp(A(tau) - B(tau) r)
  (단순화: 위험프리미엄 0, 실측=위험중립 가정)
- 세 충격(W_r, W_p, Z_s)은 상관행렬로 결합.
"""
import numpy as np
from dataclasses import dataclass


@dataclass
class EconomyV2:
    r0: float = 0.025
    r_theta: float = 0.03
    r_kappa: float = 0.15
    r_sigma: float = 0.010
    pi0: float = 0.02
    pi_theta: float = 0.02
    pi_kappa: float = 0.40
    pi_sigma: float = 0.010
    erp: float = 0.04          # 주식 위험프리미엄
    s_sigma: float = 0.18
    bond_duration: float = 5.0
    rho_rp: float = 0.5        # 금리-물가 충격 상관
    rho_rs: float = -0.2       # 금리-주식 충격 상관
    rho_ps: float = -0.1       # 물가-주식 충격 상관
    stock_shocks: dict | None = None  # 쇼크 테스트: {연차: 주식수익률}, 예 {0: -0.40}
    infl_shocks: dict | None = None   # 쇼크 테스트: {연차: 물가상승률}


def _ou_step(x, theta, kappa, sigma, z):
    """OU 정확 이산화 (dt=1)."""
    e = np.exp(-kappa)
    sd = sigma * np.sqrt((1 - e**2) / (2 * kappa))
    return theta + (x - theta) * e + sd * z


def vasicek_price(tau, r, e: EconomyV2):
    k, th, s = e.r_kappa, e.r_theta, e.r_sigma
    B = (1 - np.exp(-k * tau)) / k
    A = (th - s**2 / (2 * k**2)) * (B - tau) - s**2 * B**2 / (4 * k)
    return np.exp(A - B * r)


def generate(e: EconomyV2, T: int, n: int, rng) -> dict:
    C = np.array([[1, e.rho_rp, e.rho_rs],
                  [e.rho_rp, 1, e.rho_ps],
                  [e.rho_rs, e.rho_ps, 1]])
    L = np.linalg.cholesky(C)
    Z = rng.standard_normal((n, T, 3)) @ L.T

    r = np.empty((n, T + 1)); r[:, 0] = e.r0
    pi = np.empty((n, T + 1)); pi[:, 0] = e.pi0
    stock = np.empty((n, T)); bond = np.empty((n, T))
    D = e.bond_duration
    for t in range(T):
        r[:, t + 1] = _ou_step(r[:, t], e.r_theta, e.r_kappa, e.r_sigma, Z[:, t, 0])
        pi[:, t + 1] = _ou_step(pi[:, t], e.pi_theta, e.pi_kappa, e.pi_sigma, Z[:, t, 1])
        stock[:, t] = np.exp(r[:, t] + e.erp - 0.5 * e.s_sigma**2 + e.s_sigma * Z[:, t, 2]) - 1
        bond[:, t] = vasicek_price(D - 1, r[:, t + 1], e) / vasicek_price(D, r[:, t], e) - 1

    for t, v in (e.stock_shocks or {}).items():
        if t < T: stock[:, t] = v
    infl = pi[:, 1:].copy()
    for t, v in (e.infl_shocks or {}).items():
        if t < T: infl[:, t] = v
    cpi = np.concatenate([np.ones((n, 1)), np.cumprod(1 + infl, axis=1)], axis=1)
    return {"stock": stock, "bond": bond, "infl": infl, "cpi": cpi, "rate": r}


In [ ]:
%%writefile retire_sim/pension.py
"""한국 연금 제도 모듈 (v1: 국민연금 조기/연기 조정)."""
from .config import Person, NPSRules


def nps_annual_amount(p: Person, r: NPSRules) -> float:
    """조기/연기 반영한 국민연금 연액(현재가치, 물가연동)."""
    diff = p.nps_start_age - p.nps_normal_age
    if diff < 0:
        diff = max(diff, -r.max_early_years)
        factor = 1 + r.early_cut_per_year * diff
    else:
        diff = min(diff, r.max_defer_years)
        factor = 1 + r.defer_add_per_year * diff
    return p.nps_monthly * 12 * factor


In [ ]:
%%writefile retire_sim/engine.py
"""연간 현금흐름 엔진. 모든 경로를 벡터화해 한 번에 계산."""
import numpy as np
from .config import Household, SimConfig
from . import economy, mortality, pension


def run(hh: Household, cfg: SimConfig, qx_table: dict | None = None, economy_v2=None) -> dict:
    rng = np.random.default_rng(cfg.seed)
    qx_table = qx_table or mortality.gompertz_qx(cfg.max_age)
    youngest = min(m.age for m in hh.members)
    T = cfg.max_age - youngest
    n = cfg.n_paths

    if economy_v2 is not None:
        from . import economy_v2 as ev2
        eco = ev2.generate(economy_v2, T, n, rng)
    else:
        eco = economy.generate(cfg.economy, T, n, rng)
    cpi = eco["cpi"]

    alive = np.stack([mortality.simulate_alive(qx_table[m.sex], m.age, T, n, rng)
                      for m in hh.members])            # (k, n, T+1)
    n_alive = alive.sum(axis=0)                        # (n, T+1)
    hh_alive = n_alive > 0

    # 간병비 상태
    care_left = np.zeros((len(hh.members), n), dtype=int)
    care_cost = np.zeros((len(hh.members), n))

    W = np.zeros((n, T + 1))
    W[:, 0] = hh.liquid_assets
    depleted_at = np.full(n, -1)
    comp = {k: np.zeros((n, T)) for k in ["income", "spending", "care"]}

    for t in range(T):
        # 1) 수입
        income = np.zeros(n)
        for i, m in enumerate(hh.members):
            age = m.age + t
            a = alive[i, :, t]
            if age >= m.nps_start_age:
                income += a * pension.nps_annual_amount(m, cfg.nps) * cpi[:, t]
            if m.private_pension_start <= age < m.private_pension_start + m.private_pension_years:
                income += a * m.private_pension_annual      # 명목 고정
        # 2) 지출
        ratio = np.where(n_alive[:, t] >= 2, 1.0, hh.survivor_spending_ratio)
        spend = hh.annual_spending * ratio * cpi[:, t] * hh_alive[:, t]
        # 3) 간병비 점프
        care = np.zeros(n)
        if cfg.care.enabled:
            c = cfg.care
            for i, m in enumerate(hh.members):
                a = alive[i, :, t]
                new = a & (care_left[i] == 0) & (m.age + t >= c.start_age) & (rng.random(n) < c.lam)
                care_cost[i] = np.where(new, c.cost_median * np.exp(c.cost_log_sigma * rng.standard_normal(n)), care_cost[i])
                care_left[i] = np.where(new, c.duration_years, care_left[i])
                active = a & (care_left[i] > 0)
                care += active * care_cost[i] * cpi[:, t]
                care_left[i] = np.where(active, care_left[i] - 1, 0)
        # 4) 자산 갱신: 연초 순인출 후 수익률 적용
        net = income - spend - care
        w = np.maximum(W[:, t] + net, 0)
        newly = (W[:, t] + net <= 0) & (depleted_at < 0) & hh_alive[:, t]
        depleted_at[newly] = t
        r = hh.stock_weight * eco["stock"][:, t] + (1 - hh.stock_weight) * eco["bond"][:, t]
        W[:, t + 1] = w * (1 + r)
        comp["income"][:, t], comp["spending"][:, t], comp["care"][:, t] = income, spend, care

    last_alive_t = hh_alive.sum(axis=1) - 1
    return {"W_nominal": W, "W_real": W / cpi, "cpi": cpi, "alive": alive,
            "hh_alive": hh_alive, "depleted_at": depleted_at,
            "last_alive_t": last_alive_t, "youngest_age": youngest, "T": T, "rate": eco.get("rate"), "infl": eco["infl"], **comp}


In [ ]:
%%writefile retire_sim/metrics.py
"""결과 지표."""
import numpy as np


def summarize(res: dict) -> dict:
    d, y = res["depleted_at"], res["youngest_age"]
    dep = d >= 0
    ages = d[dep] + y
    W_real = res["W_real"]
    p = float(dep.mean()); se = float(np.sqrt(p * (1 - p) / len(d)))
    return {
        "고갈확률(생존 중)": p,
        "95% 신뢰구간 ±": 1.96 * se,
        "고갈 나이 중앙값(최연소 기준)": float(np.median(ages)) if dep.any() else None,
        "고갈 나이 10%분위": float(np.percentile(ages, 10)) if dep.any() else None,
        "가구 존속기간 중앙값(년)": float(np.median(res["last_alive_t"] + 1)),
        "10년 후 실질자산 중앙값(만원)": float(np.median(W_real[:, 10])),
    }


def depletion_curve(res: dict) -> tuple:
    """나이별 누적 고갈확률."""
    T, y, d = res["T"], res["youngest_age"], res["depleted_at"]
    ts = np.arange(T + 1)
    cum = np.array([(d >= 0) & (d <= t) for t in ts]).mean(axis=1)
    return ts + y, cum


def fan(res: dict, pct=(5, 25, 50, 75, 95)) -> tuple:
    """생존 가구 기준 실질자산 분위수."""
    W, alive = res["W_real"], res["hh_alive"]
    out = []
    for t in range(W.shape[1]):
        v = W[alive[:, t], t]
        out.append(np.percentile(v, pct) if v.size > 100 else [np.nan] * len(pct))
    return np.arange(W.shape[1]) + res["youngest_age"], np.array(out)


In [ ]:
%%writefile retire_sim/tax.py
"""한국 세금·건보료 모듈 (2026-09 기준, 단순화 버전 — 세무 자문 아님).

반영: 종합소득세(공적연금 + 금융소득종합과세 비교과세), 사적연금 분리과세(1,500만원),
      지역가입자 건강보험료(소득분만), 상속세(2차 상속), ISA·연금계좌 절세 전략.
미반영: 배당세액공제(Gross-up), 고배당기업 분리과세 특례, 재산분 건보료, 해외주식 양도세,
        연금계좌 세액공제(보수적으로 미적용), 기초연금, 주택·부동산.
금액 단위: 만원(명목). 세법 기준금액은 물가연동하지 않음(실제 법과 동일).
"""
import numpy as np
from dataclasses import dataclass

# 종합소득세 누진세율 (과세표준 상한, 세율, 누진공제)
BRACKETS = [(1400, .06, 0), (5000, .15, 126), (8800, .24, 576), (15000, .35, 1544),
            (30000, .38, 1994), (50000, .40, 2594), (100000, .42, 3594), (np.inf, .45, 6594)]
ESTATE = [(10000, .10, 0), (50000, .20, 1000), (100000, .30, 6000), (300000, .40, 16000), (np.inf, .50, 46000)]


def progressive(base, table):
    base = np.maximum(base, 0); out = np.zeros_like(base, dtype=float)
    lo = 0
    for hi, r, ded in table:
        m = (base > lo) & (base <= hi)
        out = np.where(m, base * r - ded, out); lo = hi
    return out


def pension_deduction(x):
    """연금소득공제 (한도 900만원)."""
    x = np.maximum(x, 0)
    d = np.where(x <= 350, x, np.where(x <= 700, 350 + .4 * (x - 350),
         np.where(x <= 1400, 490 + .2 * (x - 700), 630 + .1 * (x - 1400))))
    return np.minimum(d, 900)


def private_pension_rate(age):
    return np.where(age >= 80, .033, np.where(age >= 70, .044, .055))


@dataclass
class TaxConfig:
    enabled: bool = True
    nps_taxable_ratio: float = 0.75   # 국민연금 중 과세 대상 비율(2002년 이후 납입분) — 공단 원천징수 내역으로 확인
    div_yield: float = 0.02           # 주식 배당수익률(수익률 안에 포함, 과세계좌에서 과세)
    dependent_possible: bool = False  # 직장가입자 자녀 등의 피부양자 등록 가능 여부
    ownership: tuple = None           # 금융자산 명의 비율, None이면 균등
    overseas_share: float = 0.0       # 주식 중 해외주식 비율(양도차익 250만원 공제 후 22%)
    n_children: int = 2
    gift_per_child_10y: float = 0     # 전략: 10년마다 자녀 1인당 증여액(만원, 현재가치). 5,000까지 증여세 없음
    use_isa: bool = False             # 전략: 과세계좌 → ISA (1인 연 2,000, 총 1억)
    use_pension: bool = False         # 전략: 과세계좌 → 연금저축 (1인 연 1,800), 55세·가입 5년 후 연 1,500 이내 인출
    isa_annual: float = 2000; isa_total: float = 10000; isa_free_per_year: float = 200 / 3
    pen_annual: float = 1800; pen_withdraw_cap: float = 1500; pen_wait_years: int = 5
    hi_rate: float = 0.0719; ltc_ratio: float = 0.9448 / 7.19


def income_tax_person(nps, fin, age, tc: TaxConfig, extra=0):
    """공적연금 + 금융소득(비교과세) + 기타 종합소득(임대 등) 종합소득세, 지방세 포함."""
    other = np.maximum(nps * tc.nps_taxable_ratio - pension_deduction(nps * tc.nps_taxable_ratio), 0) + extra
    base = np.maximum(other - 150 - np.where(age >= 70, 100, 0), 0)
    sep = progressive(base, BRACKETS) + fin * .14
    comb = progressive(base + np.maximum(fin - 2000, 0), BRACKETS) + np.minimum(fin, 2000) * .14
    return np.where(fin > 2000, np.maximum(sep, comb), sep) * 1.1


def private_pension_tax(taxable, age):
    return np.where(taxable <= 1500, taxable * private_pension_rate(age), taxable * .165)


def health_premium(nps_list, fin_list, tc: TaxConfig, rent_list=None, prop_base_list=None, prop_monthly=0.0):
    """지역가입자 건보료(가구): 소득분(공적연금 50%, 금융소득 1,000만원 초과 시 전액, 임대소득금액 100%)
    + 재산분(공단 모의계산값 입력). 피부양자: 소득 2,000만원·재산과표 5.4억/9억·과세 임대소득 기준."""
    z = np.zeros_like(nps_list[0])
    rent_list = rent_list or [z] * len(nps_list); prop_base_list = prop_base_list or [z] * len(nps_list)
    inc_fin = [np.where(f > 1000, f, 0) for f in fin_list]
    if tc.dependent_possible:
        ok = np.ones_like(z, dtype=bool)
        for n, f, r, pb in zip(nps_list, inc_fin, rent_list, prop_base_list):
            inc = n + f + r * .5
            ok &= (inc <= 2000) & (r <= 0) & (pb <= 90000) & ~((pb > 54000) & (inc > 1000))
    else:
        ok = np.zeros_like(z, dtype=bool)
    base = sum(.5 * n + f + r * .5 for n, f, r in zip(nps_list, inc_fin, rent_list))
    return np.where(ok, 0, base * tc.hi_rate * (1 + tc.ltc_ratio) + prop_monthly * 12)


def estate_tax(estate, has_spouse):
    """상속세 (금융재산 기준). 배우자 생존 시 배우자공제 최소 5억 추가."""
    fin_ded = np.where(estate <= 2000, estate, np.where(estate <= 10000, 2000, np.minimum(estate * .2, 20000)))
    ded = 50000 + fin_ded + np.where(has_spouse, 50000, 0)
    return progressive(estate - ded, ESTATE)


# ───────────── v7: 부동산·임대·해외주식·증여 (근사, 2026-09 기준) ─────────────
PROP_TAX = [(6000, .001, 0), (15000, .0015, 3), (30000, .0025, 18), (np.inf, .004, 63)]          # 주택 재산세(일반)
PROP_TAX_1H = [(6000, .0005, 0), (15000, .001, 3), (30000, .002, 18), (np.inf, .0035, 63)]       # 1주택 특례(공시 9억 이하)
CJS_2 = [(30000, .005, 0), (60000, .007, 60), (120000, .010, 240), (250000, .013, 600),
         (500000, .015, 1100), (940000, .020, 3600), (np.inf, .027, 10180)]                        # 종부세 2주택 이하
CJS_3 = [(30000, .005, 0), (60000, .007, 60), (120000, .010, 240), (250000, .020, 1080),
         (500000, .030, 3580), (940000, .040, 8580), (np.inf, .050, 17980)]                        # 종부세 3주택 이상


@dataclass
class HouseConfig:
    official: float = 0          # 보유 주택 공시가격 합계(만원)
    market: float = 0            # 시세 합계(만원) — 상속재산 평가용
    n_houses: int = 1
    owner_share: tuple = None    # 주택 명의 비율(구성원 순서), None이면 본인 100%
    years_held: int = 10         # 보유 기간(종부세 장기보유공제)
    hi_property_monthly: float = None  # 건보료 재산분(월, 만원) — 공단 모의계산값. None이면 0으로 두고 경고
    rent_annual: float = 0       # 연 임대수입(만원)
    fmv_ratio_prop: float = .60  # 재산세 공정시장가액비율(1주택 특례는 43~45%) — 근사
    fmv_ratio_cjs: float = .60   # 종부세 공정시장가액비율


def property_tax(official, n_houses, fmv):
    base = official * (0.45 if n_houses == 1 else fmv)
    tbl = PROP_TAX_1H if (n_houses == 1 and np.all(official <= 90000)) else PROP_TAX
    t = progressive(base, tbl)
    return t * 1.2 + base * .0014                      # 지방교육세 20% + 도시지역분 0.14%


def comprehensive_property_tax(official, n_houses, age, years_held, fmv=.60):
    ded = 120000 if n_houses == 1 else 90000
    base = np.maximum(official - ded, 0) * fmv
    t = progressive(base, CJS_2 if n_houses <= 2 else CJS_3)
    if n_houses == 1:                                    # 1주택 고령자·장기보유 세액공제(합산 80% 한도)
        a = 0.4 if age >= 70 else 0.3 if age >= 65 else 0.2 if age >= 60 else 0
        h = 0.5 if years_held >= 15 else 0.4 if years_held >= 10 else 0.2 if years_held >= 5 else 0
        t = t * (1 - min(a + h, 0.8))
    return t * 1.2                                       # 농어촌특별세 20%


def rent_tax(rent_person, other_income_person):
    """주택임대소득: 2,000만원 이하 분리과세(필요경비 50%, 기본공제 200만원) 근사. 초과분은 종합과세용 소득금액 반환."""
    sep = np.maximum(rent_person * .5 - np.where(other_income_person <= 2000, 200, 0), 0) * .14 * 1.1
    return np.where(rent_person <= 2000, sep, 0), np.where(rent_person > 2000, rent_person * .5, 0)


def gift_tax(amount_over_deduction):
    return progressive(amount_over_deduction, ESTATE)


In [ ]:
%%writefile retire_sim/engine_tax.py
"""세금·계좌·부동산 반영 엔진 (v7).

계좌: 과세계좌(Wt, 취득가 Bt 추적) / ISA(Wi) / 연금계좌(Wp, 원금 Pp)
주택: 유동성 없음(생활비로 못 씀), 물가만큼 가치 상승, 보유세·건보 재산분·상속재산에 반영
순서(매년): 연금수입 → 증여 → 계좌 이전 → 지출 → 연금계좌 인출 → 세금·건보료 → 과세계좌·ISA 인출 → 수익률 → 상속
"""
import numpy as np
from .config import Household, SimConfig
from . import mortality, pension
from .economy_v2 import EconomyV2, generate
from .tax import (TaxConfig, HouseConfig, income_tax_person, private_pension_tax, health_premium, estate_tax,
                  property_tax, comprehensive_property_tax, rent_tax, gift_tax)


def run(hh: Household, cfg: SimConfig, e: EconomyV2 = None, tc: TaxConfig = None,
        house: HouseConfig = None, qx_table=None, buffer_years=5, biz=None):
    """biz: 구성원별 사업 정보 리스트 [dict(income=연 사업소득(만원), until_age=폐업 나이, workplace=직장가입자 여부)] 또는 None"""
    e, tc, house = e or EconomyV2(), tc or TaxConfig(), house or HouseConfig()
    biz = biz or [None] * len(hh.members)
    rng = np.random.default_rng(cfg.seed)
    qx_table = qx_table or mortality.gompertz_qx(cfg.max_age)
    k = len(hh.members); youngest = min(m.age for m in hh.members)
    T, n = cfg.max_age - youngest, cfg.n_paths
    eco = generate(e, T, n, rng); cpi = eco["cpi"]
    alive = np.stack([mortality.simulate_alive(qx_table[m.sex], m.age, T, n, rng) for m in hh.members])
    hh_alive = alive.sum(0) > 0
    own = np.array(tc.ownership if tc.ownership else [1 / k] * k, float)
    hown = np.array(house.owner_share if house.owner_share else [1.0] + [0.0] * (k - 1), float)

    Wt = np.full(n, float(hh.liquid_assets)); Bt = Wt.copy(); Wi = np.zeros(n); Wp = np.zeros(n); Pp = np.zeros(n)
    isa_in = np.zeros(n); cg_due = np.zeros(n)
    gifts_hist = []                                   # (t, 명목 증여액, 증여세)
    gift_val = np.zeros(n)                          # 증여한 돈의 현재 가치(자녀도 같은 수익률로 운용 가정, 명목)
    care_left = np.zeros((k, n), int); care_cost = np.zeros((k, n))
    dep_at = np.full(n, -1)
    tax_y = np.zeros((n, T)); hi_y = np.zeros((n, T)); prop_y = np.zeros((n, T))
    estate_real = np.full(n, np.nan); etax_real = np.full(n, np.nan); transfer_real = np.full(n, np.nan)
    w = hh.stock_weight; W_hist = np.zeros((n, T + 1)); W_hist[:, 0] = Wt

    def sell_from_taxable(amount):
        """과세계좌에서 매도: 실현이익 반환(해외주식분만 과세 대상)."""
        nonlocal Wt, Bt
        amt = np.minimum(np.maximum(amount, 0), Wt)
        ratio = np.where(Wt > 0, Bt / np.maximum(Wt, 1e-9), 1)
        gain = amt * np.maximum(1 - ratio, 0)
        Bt = np.maximum(Bt - amt * np.minimum(ratio, 1), 0); Wt = Wt - amt
        return amt, gain * w * tc.overseas_share

    for t in range(T):
        a = alive[:, :, t]; na = a.sum(0); live = na > 0
        # 명의 재분배(사망자 몫은 생존자에게)
        def redistribute(base):
            sh = base[:, None] * a; orphan = sh.sum(0) == 0
            sh = np.where(orphan[None, :], a, sh); return sh / np.maximum(sh.sum(0), 1e-9)
        fshare, hshare = redistribute(own), redistribute(hown)

        # 1) 연금 수입
        nps = np.zeros((k, n)); priv = np.zeros((k, n))
        for i, m in enumerate(hh.members):
            age = m.age + t
            if age >= m.nps_start_age: nps[i] = a[i] * pension.nps_annual_amount(m, cfg.nps) * cpi[:, t]
            if m.private_pension_start <= age < m.private_pension_start + m.private_pension_years:
                priv[i] = a[i] * m.private_pension_annual
        bz = np.zeros((k, n)); work = np.zeros((k, n), bool)
        for i, m in enumerate(hh.members):
            b = biz[i]
            if b and m.age + t < b["until_age"]:
                bz[i] = a[i] * b["income"] * cpi[:, t]
                work[i] = a[i] & bool(b.get("workplace", False))
        rent = house.rent_annual * cpi[:, t] * live
        official = house.official * cpi[:, t]

        # 2) 증여 전략 (10년마다)
        realized = np.zeros(n); gift_tax_now = np.zeros(n)
        if tc.enabled and tc.gift_per_child_10y > 0 and t % 10 == 0:
            want = tc.gift_per_child_10y * tc.n_children * cpi[:, t] * live
            buf = np.maximum(Wt - buffer_years * hh.annual_spending * cpi[:, t], 0)
            g = np.minimum(want, buf)
            amt, gain = sell_from_taxable(g); realized += gain
            per_child = amt / max(tc.n_children, 1)
            gtax = gift_tax(np.maximum(per_child - 5000, 0)) * tc.n_children
            gifts_hist.append((t, amt, gtax)); gift_val += amt - gtax; gift_tax_now = gtax

        # 3) 계좌 이전 (생활비 버퍼 유지)
        buf = np.maximum(Wt - buffer_years * hh.annual_spending * cpi[:, t], 0)
        if tc.enabled and tc.use_isa:
            room = np.minimum(tc.isa_annual * na, np.maximum(tc.isa_total * na - isa_in, 0))
            amt, gain = sell_from_taxable(np.minimum(room, buf) * live); realized += gain
            Wi += amt; isa_in += amt; buf -= amt
        if tc.enabled and tc.use_pension:
            amt, gain = sell_from_taxable(np.minimum(tc.pen_annual * na, buf) * live); realized += gain
            Wp += amt; Pp += amt

        # 4) 과세계좌 금융소득
        fin_tot = Wt * (w * tc.div_yield + (1 - w) * np.maximum(eco["rate"][:, t], 0))
        fin_i = [fshare[i] * fin_tot for i in range(k)]

        # 5) 지출·간병비
        ratio = np.where(na >= 2, 1.0, hh.survivor_spending_ratio)
        spend = hh.annual_spending * ratio * cpi[:, t] * live
        care = np.zeros(n)
        if cfg.care.enabled:
            c = cfg.care
            for i, m in enumerate(hh.members):
                new = a[i] & (care_left[i] == 0) & (m.age + t >= c.start_age) & (rng.random(n) < c.lam)
                care_cost[i] = np.where(new, c.cost_median * np.exp(c.cost_log_sigma * rng.standard_normal(n)), care_cost[i])
                care_left[i] = np.where(new, c.duration_years, care_left[i])
                act = a[i] & (care_left[i] > 0); care += act * care_cost[i] * cpi[:, t]
                care_left[i] = np.where(act, care_left[i] - 1, 0)

        # 6) 연금계좌 인출
        need = spend + care - nps.sum(0) - priv.sum(0) - rent - bz.sum(0)
        pw = np.zeros(n); pw_taxable = np.zeros(n)
        if tc.enabled and tc.use_pension and t >= tc.pen_wait_years:
            pw = np.clip(np.minimum(need, tc.pen_withdraw_cap * na), 0, Wp)
            frac = np.where(Wp > 0, np.clip(1 - Pp / np.maximum(Wp, 1e-9), 0, 1), 0)
            Pp = np.maximum(Pp - pw * (1 - frac), 0); Wp -= pw; pw_taxable = pw * frac

        # 7) 세금·건보료·보유세
        taxes = np.zeros(n); hi = np.zeros(n); prop = np.zeros(n)
        if tc.enabled:
            prop_base = [hshare[i] * official * (0.45 if house.n_houses == 1 else house.fmv_ratio_prop) for i in range(k)]
            rent_i = [hshare[i] * rent for i in range(k)]
            for i, m in enumerate(hh.members):
                age = m.age + t
                sep_rent, comb_rent = rent_tax(rent_i[i], nps[i] * tc.nps_taxable_ratio)
                taxes += a[i] * (income_tax_person(nps[i], fin_i[i], age, tc, extra=comb_rent + bz[i]) + sep_rent)
                taxes += a[i] * private_pension_tax(priv[i] + pw_taxable * fshare[i], age)
            if tc.use_isa:
                taxes += np.maximum(Wi * (w * tc.div_yield + (1 - w) * np.maximum(eco["rate"][:, t], 0))
                                    - tc.isa_free_per_year * na, 0) * .099
            # 해외주식 양도세(작년 실현분, 1인 250만원 공제) + 올해 증여세
            taxes += np.maximum(cg_due - 250 * na, 0) * .22 * 1.1 * live 
            if house.official > 0:
                oldest = max(m.age for m in hh.members) + t
                # 재산세: 물건별(가구 합산 근사) / 종부세: 인별 과세(명의 비율대로 1인 공제 9억)
                prop = property_tax(official, house.n_houses, house.fmv_ratio_prop)
                co_owned = (hshare > 0).sum(0) >= 2
                per_person = sum(a[i] * comprehensive_property_tax(hshare[i] * official, max(house.n_houses, 2),
                                 hh.members[i].age + t, house.years_held + t, house.fmv_ratio_cjs) for i in range(k))
                if house.n_houses == 1:   # 1주택: 단독명의 12억 공제+고령자·장기보유 공제 / 공동명의는 1인 9억 공제와 특례 중 유리한 쪽
                    single = comprehensive_property_tax(official, 1, oldest, house.years_held + t, house.fmv_ratio_cjs)
                    cjs = np.where(co_owned, np.minimum(per_person, single), single)
                else:
                    cjs = per_person
                prop = (prop + cjs) * live
            pm = (house.hi_property_monthly or 0) * cpi[:, t]
            rate = tc.hi_rate * (1 + tc.ltc_ratio)
            any_work = work.any(0)
            # 직장가입자(사업장 대표): 사업소득 전액(대표자 전액 부담) + 보수 외 소득 2,000만원 초과분
            hi_w = np.zeros(n)
            for i in range(k):
                other = .5 * nps[i] + np.where(fin_i[i] > 1000, fin_i[i], 0) + .5 * rent_i[i]
                hi_w += work[i] * rate * (bz[i] + np.maximum(other - 2000, 0))
            # 지역가입자(직장 아닌 생존자): 소득분 + 재산분(직장가입자 몫 주택 지분 제외)
            nonw = [a[i] & ~work[i] for i in range(k)]
            reg_inc = sum(nonw[i] * (.5 * nps[i] + np.where(fin_i[i] > 1000, fin_i[i], 0) + .5 * rent_i[i] + bz[i]) for i in range(k))
            reg_prop_share = sum(nonw[i] * hshare[i] for i in range(k))
            all_reg = health_premium([nps[i] for i in range(k)], fin_i, tc, rent_i, prop_base, pm)
            mixed_reg = reg_inc * rate + pm * 12 * reg_prop_share
            hi = (hi_w + np.where(any_work, mixed_reg, all_reg + sum(bz[i] for i in range(k)) * rate)) * live
        tax_y[:, t] = taxes / cpi[:, t]; hi_y[:, t] = hi / cpi[:, t]; prop_y[:, t] = prop / cpi[:, t]

        # 8) 인출: 과세계좌 → ISA → (부족 시) 연금계좌 연금외수령
        rest = need - pw + taxes + hi + prop
        amt, gain = sell_from_taxable(np.maximum(rest, 0)); realized += gain
        from_i = np.minimum(np.maximum(rest, 0) - amt, Wi); Wi -= from_i
        surplus = np.maximum(-rest, 0); Wt += surplus; Bt += surplus
        short = np.maximum(rest, 0) - amt - from_i
        if tc.enabled and tc.use_pension:
            frac2 = np.where(Wp > 0, np.clip(1 - Pp / np.maximum(Wp, 1e-9), 0, 1), 0)
            gross = np.minimum(short / np.maximum(1 - .165 * frac2, 1e-9), Wp)
            Pp = np.maximum(Pp - gross * (1 - frac2), 0); Wp -= gross
            tax_y[:, t] += gross * frac2 * .165 / cpi[:, t]
            short = np.maximum(short - gross * (1 - .165 * frac2), 0)
        newly = (short > 1e-6) & (dep_at < 0) & live; dep_at[newly] = t
        cg_due = realized

        # 9) 수익률
        r = w * eco["stock"][:, t] + (1 - w) * eco["bond"][:, t]
        Wt *= (1 + r); Wi *= (1 + r); Wp *= (1 + r); gift_val *= (1 + r)
        W_hist[:, t + 1] = (Wt + Wi + Wp) * (dep_at < 0)

        # 10) 마지막 생존자 사망 → 상속세(10년 내 증여 합산, 기납부 증여세 공제)
        died_all = live & (alive[:, :, t + 1].sum(0) == 0)
        if died_all.any():
            fin_est = (Wt + Wi + Wp) * (dep_at < 0)
            est = fin_est + house.market * cpi[:, t + 1]
            recent = sum(g for (tg, g, _) in gifts_hist if t + 1 - tg < 10) if gifts_hist else 0
            recent_gt = sum(gt for (tg, _, gt) in gifts_hist if t + 1 - tg < 10) if gifts_hist else 0
            fin_ded = np.where(fin_est <= 2000, fin_est, np.where(fin_est <= 10000, 2000, np.minimum(fin_est * .2, 20000)))
            lump = max(50000, 20000 + 5000 * tc.n_children)
            et = np.maximum(progressive_estate(est + recent - lump - fin_ded) - recent_gt, 0)
            estate_real = np.where(died_all, (est - et) / cpi[:, t + 1], estate_real)
            etax_real = np.where(died_all, et / cpi[:, t + 1], etax_real)
            transfer_real = np.where(died_all, (est - et + gift_val) / cpi[:, t + 1], transfer_real)

    return {"depleted_at": dep_at, "youngest_age": youngest, "T": T, "tax_y": tax_y, "hi_y": hi_y, "prop_y": prop_y,
            "estate_real": estate_real, "estate_tax_real": etax_real, "transfer_real": transfer_real,
            "W_nominal": W_hist, "W_real": W_hist / cpi, "hh_alive": hh_alive,
            "last_alive_t": hh_alive.sum(1) - 1, "cpi": cpi}


def progressive_estate(base):
    from .tax import progressive, ESTATE
    return progressive(base, ESTATE)


def summarize(res):
    d = res["depleted_at"]; p = float((d >= 0).mean()); se = float(np.sqrt(p * (1 - p) / len(d)))
    y = min(20, res["T"])
    med = lambda x: float(np.median(x[~np.isnan(x)])) if np.any(~np.isnan(x)) else 0.0
    return {"고갈확률": p, "±": 1.96 * se,
            "20년 세금": float(res["tax_y"][:, :y].sum(1).mean()),
            "20년 건보료": float(res["hi_y"][:, :y].sum(1).mean()),
            "20년 보유세": float(res["prop_y"][:, :y].sum(1).mean()),
            "상속세(중앙값)": med(res["estate_tax_real"]),
            "가족 이전 총액(중앙값)": med(res["transfer_real"])}


In [ ]:
import importlib, sys
for m in list(sys.modules):
    if m.startswith('retire_sim'): del sys.modules[m]
import numpy as np, matplotlib.pyplot as plt
from retire_sim.config import Person, Household, SimConfig, CareShock
from retire_sim.economy_v2 import EconomyV2
from retire_sim import engine, metrics, mortality, engine_tax
from retire_sim.tax import TaxConfig, HouseConfig
plt.style.use('dark_background')
print('설치 완료')

## 2단계. 실제 정보 입력 ← 이 셀만 고치면 됩니다
- 금액 단위는 모두 **만원**, 오늘 기준 금액으로 넣습니다.
- 배우자가 없으면 `SPOUSE = None`으로 두세요.
- 모르는 값은 기본값 그대로 두고, 결과를 볼 때 "추정값"임을 감안하세요.

In [ ]:
# ※ 아래 값은 예시 가구입니다. 본인 정보로 바꿔서 실행하세요. (금액 단위: 만원)
# ── 본인 ─────────────────────────────
ME = dict(
    age=60, sex="M",                  # 나이, 성별("M"/"F" 대문자)
    nps_monthly=110,                  # 국민연금 예상 월액(만원) - 국민연금공단 '내 연금 알아보기'
    nps_normal_age=65,                # 정상 수령 개시 나이(조회 화면에 표시)
    private_pension_annual=600,       # 연금저축·IRP·연금보험 예상 연 수령액(만원), 없으면 0
    private_pension_start=60,         # 사적연금 받기 시작하는 나이
    private_pension_years=20,         # 사적연금 받는 기간(년)
)
# ── 배우자 (없으면 SPOUSE = None) ─────
SPOUSE = dict(
    age=58, sex="F", nps_monthly=50, nps_normal_age=65,
    private_pension_annual=0, private_pension_start=0, private_pension_years=0,
)
# ── 가구 ──────────────────────────────
LIQUID_ASSETS = 50_000    # 금융자산 합계(만원): 예금+주식+펀드+ETF 등, 부동산 제외
STOCK_WEIGHT  = 0.4       # 금융자산 중 주식·주식형펀드 비율 (0~1)
SPENDING      = 3600      # 연 생활비(만원) = 월 생활비 × 12. 보험료 포함, 보유세·건보료는 제외(모델이 따로 계산)
SURVIVOR_RATIO = 0.7      # 한 명 사망 후 생활비 비율
NPS_START     = 65        # 국민연금 실제 개시 나이(조기면 작게, 연기면 크게)

# ── 세금 관련 (9단계에서 사용) ────────
NPS_TAXABLE_RATIO = 0.75   # 국민연금 중 과세 대상 비율(2002년 이후 납입분, 추정)
DIV_YIELD = 0.02           # 보유 주식의 배당수익률(대략)
DEPENDENT_POSSIBLE = False # 직장 다니는 자녀 밑으로 피부양자 등록이 가능하면 True
OWNERSHIP = None           # 금융자산 명의 비율. None=부부 균등, (1, 0)=본인 명의 집중
OVERSEAS_SHARE = 0.2       # 주식 중 해외주식 비율 (0~1)

# ── 부동산·임대 (없으면 0) ────────────
HOUSE_OFFICIAL = 60_000    # 보유 주택 공시가격 합계(만원) - 부동산공시가격알리미
HOUSE_MARKET   = 90_000    # 보유 주택 시세 합계(만원) - 상속세 계산용
N_HOUSES       = 1         # 보유 주택 수
HOUSE_OWNER    = None      # 주택 명의 비율. None=본인 100%, (0.5, 0.5)=부부 공동
YEARS_HELD     = 15        # 주택 보유 기간(년)
HI_PROPERTY_MONTHLY = 12   # (지역가입자) 건보료 재산분(월, 만원) - 공단 '지역보험료 모의계산'에서 소득 0, 재산만 넣은 값
RENT_ANNUAL    = 0         # 연 임대수입(만원)

# ── 사업 (없으면 None) ────────────────
# income: 연 사업소득금액(만원) / until_age: 폐업 나이 / workplace: 직원 있어 직장가입자면 True
ME_BIZ     = None          # 예: dict(income=800, until_age=65, workplace=True)
SPOUSE_BIZ = None

# ── 가족·증여 ─────────────────────────
N_CHILDREN = 2             # 자녀 수

# ── 입력 점검 ─────────────────────────
assert 0 <= STOCK_WEIGHT <= 1, "주식 비중은 0~1 사이"
for d in [ME] + ([SPOUSE] if SPOUSE else []):
    assert d["sex"] in ("M", "F"), "성별은 대문자 M 또는 F"
    assert d["private_pension_annual"] == 0 or d["private_pension_years"] > 0, "사적연금이 있으면 받는 기간(년)을 넣어주세요"
if HOUSE_MARKET > 0 and HOUSE_OFFICIAL / HOUSE_MARKET > 0.8:
    print(f"⚠️ 공시가격이 시세의 {HOUSE_OFFICIAL/HOUSE_MARKET:.0%}입니다. 보통 60~70%대라 두 값을 다시 확인해 보세요.")
if N_HOUSES >= 2 and RENT_ANNUAL == 0:
    print("ℹ️ 주택이 2채 이상인데 임대수입이 0입니다. 월세를 받고 있다면 RENT_ANNUAL에 넣어주세요.")
for nm, b in (("본인", ME_BIZ), ("배우자", SPOUSE_BIZ)):
    if b and b["income"] is None:
        print(f"⚠️ {nm} 사업소득이 비어 있어 0으로 계산합니다."); b["income"] = 0
if HOUSE_OFFICIAL > 0 and not HI_PROPERTY_MONTHLY:
    print("⚠️ 주택이 있는데 폐업 후 건보료 재산분이 0/비어 있습니다. 지금은 직장가입자라 재산분이 없어도, "
          "폐업 후 지역가입자가 되면 생깁니다. 공단 모의계산 값을 넣어야 정확합니다.")
print(f"월 생활비 {SPENDING/12:.0f}만원, 금융자산 {LIQUID_ASSETS/10000:.1f}억, "
      f"금융자산으로 버티는 기간(단순계산) 약 {LIQUID_ASSETS/max(SPENDING,1):.0f}년")

In [ ]:
def make_household(nps_start=None, spending=None, stock_weight=None):
    ns = nps_start or NPS_START
    mk = lambda d: Person(age=d["age"], sex=d["sex"], nps_monthly=d["nps_monthly"],
                          nps_normal_age=d["nps_normal_age"], nps_start_age=ns,
                          private_pension_annual=d["private_pension_annual"],
                          private_pension_start=d["private_pension_start"],
                          private_pension_years=d["private_pension_years"])
    members = [mk(ME)] + ([mk(SPOUSE)] if SPOUSE else [])
    return Household(members=members, liquid_assets=LIQUID_ASSETS,
                     stock_weight=STOCK_WEIGHT if stock_weight is None else stock_weight,
                     annual_spending=spending or SPENDING, survivor_spending_ratio=SURVIVOR_RATIO)

cfg = SimConfig(n_paths=10_000, seed=42)   # 같은 seed = 같은 1만 개 시나리오 (공통 난수)
eco = EconomyV2()                           # 금리·물가·주식 가정 (임시값)

def run(hh=None, e=None, c=None, q=None):
    return engine.run(hh or make_household(), c or cfg, qx_table=q, economy_v2=e or eco)

def show(label, res):
    s = metrics.summarize(res)
    p, ci = s['고갈확률(생존 중)'] * 100, s['95% 신뢰구간 ±'] * 100
    print(f"{label:<28} 고갈확률 {p:5.1f}% (±{ci:.1f}%p)  ≈ 10번 중 {round(p/10)}번")
    return p

## 3단계. 기본 결과

In [ ]:
res = run()
for k, v in metrics.summarize(res).items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ages, F = metrics.fan(res)
ax[0].fill_between(ages, F[:, 0], F[:, 4], alpha=.2, color='#4fc3f7', label='5-95%')
ax[0].fill_between(ages, F[:, 1], F[:, 3], alpha=.4, color='#4fc3f7', label='25-75%')
ax[0].plot(ages, F[:, 2], color='#4fc3f7', label='median')
ax[0].set(title='Real wealth of surviving households (10k KRW)', xlabel='Age (youngest)'); ax[0].legend()
a, cum = metrics.depletion_curve(res)
ax[1].plot(a, cum * 100, color='#ff8a65')
ax[1].set(title='Cumulative depletion probability (%)', xlabel='Age (youngest)')
plt.tight_layout(); plt.show()

## 4단계. 선택지 비교 ① — 국민연금 개시 나이
같은 1만 개 시나리오로 비교합니다. 신뢰구간이 겹치면 '차이 없음'으로 해석하세요.

In [ ]:
for age in (NPS_START - 5, NPS_START, NPS_START + 5):
    show(f"국민연금 {age}세 개시", run(make_household(nps_start=age)))

## 5단계. 금리·물가 시나리오

In [ ]:
show("기본 (시작 금리 2.5%)", run())
show("저금리 은퇴 (시작 금리 1.5%)", run(e=EconomyV2(r0=0.015)))
show("고금리 은퇴 (시작 금리 4.0%)", run(e=EconomyV2(r0=0.04)))
show("고물가 지속 (금리 고정)", run(e=EconomyV2(pi0=0.04, pi_theta=0.035)))
show("고물가 지속 (금리 동반 상승)", run(e=EconomyV2(pi0=0.04, pi_theta=0.035, r_theta=0.045)))

## 6단계. 쇼크 테스트
특정 해에 충격을 강제로 넣습니다. `{0: -0.40}` = 은퇴 첫해 주가 −40%.

In [ ]:
base = show("기준", run())
show("은퇴 첫해 주가 -40%", run(e=EconomyV2(stock_shocks={0: -0.40})))
show("10년 뒤 주가 -40%", run(e=EconomyV2(stock_shocks={10: -0.40})))
show("3년 연속 물가 6%", run(e=EconomyV2(infl_shocks={0: .06, 1: .06, 2: .06})))
q = mortality.gompertz_qx(); q_long = {k: v * 0.8 for k, v in q.items()}
for v in q_long.values(): v[-1] = 1
show("장수 (사망률 20% 감소)", run(q=q_long))
show("간병 위험 2배", run(c=SimConfig(care=CareShock(lam=0.06))))

## 7단계. 민감도 분석 (토네이도 차트)

In [ ]:
base = metrics.summarize(run())['고갈확률(생존 중)'] * 100
P = lambda r: metrics.summarize(r)['고갈확률(생존 중)'] * 100
tests = [
  ("Spending -10%/+10%", P(run(make_household(spending=SPENDING*0.9))), P(run(make_household(spending=SPENDING*1.1)))),
  ("Long-run rate 2%/4%", P(run(e=EconomyV2(r_theta=.02))), P(run(e=EconomyV2(r_theta=.04)))),
  ("Equity weight -20%p/+20%p", P(run(make_household(stock_weight=max(0, STOCK_WEIGHT-.2)))), P(run(make_household(stock_weight=min(1, STOCK_WEIGHT+.2))))),
  ("Inflation 1.5%/2.5%", P(run(e=EconomyV2(pi_theta=.015))), P(run(e=EconomyV2(pi_theta=.025)))),
  ("Equity premium 3%/5%", P(run(e=EconomyV2(erp=.03))), P(run(e=EconomyV2(erp=.05)))),
  ("Equity vol 15%/21%", P(run(e=EconomyV2(s_sigma=.15))), P(run(e=EconomyV2(s_sigma=.21)))),
]
tests.sort(key=lambda t: max(abs(t[1] - base), abs(t[2] - base)))
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (n, lo, hi) in enumerate(tests):
    ax.barh(i, lo - base, left=base, color='#4fc3f7'); ax.barh(i, hi - base, left=base, color='#ff8a65')
ax.axvline(base, color='w', lw=.8); ax.set_yticks(range(len(tests))); ax.set_yticklabels([t[0] for t in tests])
ax.set(title=f'Sensitivity (base {base:.1f}%) - blue: low, orange: high', xlabel='Depletion probability (%)')
plt.tight_layout(); plt.show()

## 8단계. 검증 — 시뮬레이션이 이론값과 맞는가

In [ ]:
from retire_sim.economy_v2 import generate, vasicek_price
rng = np.random.default_rng(0); e = EconomyV2()
d = generate(e, 60, 20000, rng)
print(f"금리 장기평균 {d['rate'][:, -1].mean():.4f} (이론 {e.r_theta}), 표준편차 {d['rate'][:, -1].std():.4f} (이론 {e.r_sigma/np.sqrt(2*e.r_kappa):.4f})")
print(f"물가 장기평균 {d['infl'][:, -1].mean():.4f} (이론 {e.pi_theta}), 표준편차 {d['infl'][:, -1].std():.4f} (이론 {e.pi_sigma/np.sqrt(2*e.pi_kappa):.4f})")
dt, tau, m = 1/52, 5.0, 40000; r = np.full(m, e.r0); integ = np.zeros(m)
for _ in range(int(tau/dt)):
    ek = np.exp(-e.r_kappa*dt)
    rn = e.r_theta + (r-e.r_theta)*ek + e.r_sigma*np.sqrt((1-ek**2)/(2*e.r_kappa))*rng.standard_normal(m)
    integ += .5*(r+rn)*dt; r = rn
print(f"5년 채권가격 해석해 {vasicek_price(tau, e.r0, e):.5f} vs 몬테카를로 {np.exp(-integ).mean():.5f}")
print(f"60세 남 기대여명 {mortality.life_expectancy(mortality.gompertz_qx()['M'], 60):.1f}년 (보정 목표 23.5)")

## 9단계. 세금·건보료·보유세 반영 & 절세 전략 비교 (2026-09 세법 기준, 근사)
**반영:** 종합소득세(국민연금·금융소득종합과세·임대소득), 사적연금 분리과세, 해외주식 양도세, 재산세·종부세(1주택 고령자·장기보유 공제),
지역 건보료(소득분 + 재산분 입력값), 피부양자 소득·재산 기준, 상속세(10년 내 증여 합산), 증여세
**미반영:** 배당세액공제, 고배당 분리과세 특례, 주택 양도세(매도 가정 없음), 연금계좌 세액공제, 주택연금

- **ISA**: 과세계좌 → ISA 1인 연 2,000만원(총 1억) · **연금계좌**: 1인 연 1,800만원 이전, 5년 후 1인 연 1,500만원 이내 인출
- 두 전략 모두 과세계좌에 생활비 5년치는 남겨둠 · 주택은 생활비로 쓸 수 없는 자산으로 취급

⚠️ 참고용 비교이며 세무 자문이 아닙니다.

In [ ]:
house = HouseConfig(official=HOUSE_OFFICIAL, market=HOUSE_MARKET, n_houses=N_HOUSES,
                    owner_share=HOUSE_OWNER, years_held=YEARS_HELD,
                    hi_property_monthly=HI_PROPERTY_MONTHLY, rent_annual=RENT_ANNUAL)

def tax_cfg(**kw):
    base = dict(nps_taxable_ratio=NPS_TAXABLE_RATIO, div_yield=DIV_YIELD, dependent_possible=DEPENDENT_POSSIBLE,
                ownership=OWNERSHIP, overseas_share=OVERSEAS_SHARE, n_children=N_CHILDREN)
    base.update(kw); return TaxConfig(**base)

def run_tax(tc, hh=None, e=None, h=None):
    return engine_tax.summarize(engine_tax.run(hh or make_household(), cfg, e=e or eco, tc=tc, house=h or house,
                                                    biz=[ME_BIZ] + ([SPOUSE_BIZ] if SPOUSE else [])))

def table(rows):
    print(f"{'전략':<18}{'고갈확률':>8}{'20년 세금':>10}{'건보료':>9}{'보유세':>9}{'상속세':>10}{'가족 이전':>11}")
    out = {}
    for lab, tc in rows:
        s = run_tax(tc); out[lab] = s
        print(f"{lab:<18}{s['고갈확률']*100:>7.1f}%{s['20년 세금']:>10,.0f}{s['20년 건보료']:>9,.0f}"
              f"{s['20년 보유세']:>9,.0f}{s['상속세(중앙값)']:>10,.0f}{s['가족 이전 총액(중앙값)']:>11,.0f}")
    print("\n금액: 만원·현재가치 · 세금/건보료/보유세 = 20년 누적 평균 · 상속세/가족 이전 = 중앙값")
    print("가족 이전 = 세후 상속액 + 생전 증여분(자녀도 같은 수익률로 운용 가정) · 고갈확률 오차 약 ±0.9%p")
    return out

res_tax = table([("세금 미반영", tax_cfg(enabled=False)),
                 ("기본 (모두 과세계좌)", tax_cfg()),
                 ("ISA 활용", tax_cfg(use_isa=True)),
                 ("연금계좌 활용", tax_cfg(use_pension=True)),
                 ("ISA + 연금계좌", tax_cfg(use_isa=True, use_pension=True))]
                + ([("금융자산 명의 한쪽 집중", tax_cfg(ownership=(1, 0)))] if SPOUSE else []))

In [ ]:
labs = [l for l in res_tax if l != "세금 미반영"]
en = {"기본 (모두 과세계좌)": "Base", "ISA 활용": "ISA", "연금계좌 활용": "Pension",
      "ISA + 연금계좌": "ISA+Pension", "금융자산 명의 한쪽 집중": "Single owner"}
parts = ['20년 세금', '20년 건보료', '20년 보유세']; cols = ['#4fc3f7', '#ffb74d', '#9e9e9e']
fig, ax = plt.subplots(figsize=(8.5, 4)); bottom = np.zeros(len(labs))
for p, c, nm in zip(parts, cols, ['Income tax', 'Health premium', 'Property tax']):
    v = np.array([res_tax[l][p] for l in labs]); ax.bar([en[l] for l in labs], v, bottom=bottom, color=c, label=nm); bottom += v
ax.set(title='20-year taxes, health premiums, property taxes (10k KRW, PV)'); ax.legend()
plt.tight_layout(); plt.show()

## 10단계. 증여 전략: 가족에게 더 남기기 vs 내 노후 안전
10년마다 자녀 1인당 증여합니다(10년 5,000만원까지 증여세 없음). 사망 전 10년 내 증여분은 상속재산에 다시 합산됩니다.
**가족 이전 총액이 늘어나는 만큼 부모의 고갈확률도 올라가는지** 함께 보세요.

In [ ]:
_ = table([("ISA+연금 (증여 없음)", tax_cfg(use_isa=True, use_pension=True)),
           ("+ 자녀당 5천만원/10년", tax_cfg(use_isa=True, use_pension=True, gift_per_child_10y=5000)),
           ("+ 자녀당 1억원/10년", tax_cfg(use_isa=True, use_pension=True, gift_per_child_10y=10000))])

## 11단계. 국민연금 개시 나이 × 세금·건보료

In [ ]:
for age in (NPS_START, NPS_START + 3, NPS_START + 5):
    s = run_tax(tax_cfg(use_isa=True, use_pension=True), hh=make_household(nps_start=age))
    print(f"국민연금 {age}세 개시 (ISA+연금)  고갈확률 {s['고갈확률']*100:5.1f}%  "
          f"20년 세금+건보료 {s['20년 세금']+s['20년 건보료']:,.0f}만원  가족 이전 {s['가족 이전 총액(중앙값)']:,.0f}만원")